<a href="https://colab.research.google.com/github/mitalidaduria/nlp-payments-lab/blob/main/sklearn_Pipeline_%2B_versioning_%2B_model_registry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
# Configure Git identity
!git config --global user.name "Mitali Daduria"
!git config --global user.email "mitalidaduriaj@gmail.com"

# Clone your repository (Replace with your actual GitHub repo URL)
!git clone https://github.com/mitalidaduria/payment-fraud-ml.git
%cd payment-fraud-ml

Cloning into 'payment-fraud-ml'...
remote: Enumerating objects: 110, done.
remote: Counting objects: 100% (110/110), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 110 (delta 37), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (110/110), 41.55 KiB | 4.62 MiB/s, done.
Resolving deltas: 100% (37/37), done.
/content/payment-fraud-ml/payment-fraud-ml


In [13]:
!mkdir -p src/serving models

In [14]:
%%writefile src/serving/pipeline.py
import os
import json
import subprocess
from datetime import datetime
import joblib
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier
# 1. Custom Feature Engineering Transformer
class PaymentFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    Custom transformer to calculate domain-specific features
    (e.g., log transformation of amount to handle skewness).
    """
    def __init__(self):
        pass

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        if 'amount' in X_out.columns:
            X_out['amount_log'] = np.log1p(X_out['amount'])
        return X_out
# 2. Main FraudPipeline Implementation
class FraudPipeline:
    def __init__(self, numeric_features=None, categorical_features=None, xgb_params=None):
        self.numeric_features = numeric_features or ['amount']
        self.categorical_features = categorical_features or ['payment_type', 'device_category']
        self.xgb_params = xgb_params or {'n_estimators': 100, 'max_depth': 5, 'random_state': 42}
        self.pipeline = None

    def _get_git_hash(self) -> str:
        """Retrieves the short Git commit hash of the current repository state."""
        try:
            return subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD']).decode('utf-8').strip()
        except Exception:
            return "nogithash"

    def build(self):
        """Combines Feature Engineering + Preprocessor + Model into one serializable Pipeline."""
        numeric_transformer = Pipeline(steps=[
            ('scaler', StandardScaler())
        ])

        categorical_transformer = Pipeline(steps=[
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ])

        preprocessor = ColumnTransformer(
            transformers=[
                ('num', numeric_transformer, self.numeric_features),
                ('cat', categorical_transformer, self.categorical_features)
            ]
        )

        self.pipeline = Pipeline(steps=[
            ('domain_features', PaymentFeatureEngineer()),
            ('preprocessor', preprocessor),
            ('classifier', XGBClassifier(**self.xgb_params))
        ])
        return self.pipeline

    def fit(self, X: pd.DataFrame, y: pd.Series):
        if self.pipeline is None:
            self.build()
        self.pipeline.fit(X, y)
        return self

    def predict_proba(self, X: pd.DataFrame):
        if self.pipeline is None:
            raise ValueError("Pipeline is not fitted yet. Call fit() or load a saved model.")
        return self.pipeline.predict_proba(X)

    def save(self, output_dir: str = "models", version: str = None) -> tuple[str, str]:
        """
        Saves the fitted pipeline object using version tag: vYYYYMMDD_<git_hash[:8]>
        """
        if self.pipeline is None:
            raise ValueError("Cannot save an unfitted pipeline.")

        os.makedirs(output_dir, exist_ok=True)

        if version is None:
            date_str = datetime.now().strftime("%Y%m%d")
            git_hash = self._get_git_hash()[:8]
            version = f"v{date_str}_{git_hash}"

        file_path = os.path.join(output_dir, f"{version}.pkl")
        joblib.dump(self.pipeline, file_path)
        print(f"Pipeline saved successfully to: {file_path}")
        return version, file_path

    @classmethod
    def load(cls, filepath: str):
        """Loads a pre-trained versioned pipeline object from disk."""
        instance = cls()
        instance.pipeline = joblib.load(filepath)
        print(f"Pipeline loaded from: {filepath}")
        return instance


Writing src/serving/pipeline.py


In [15]:
%%writefile models/registry.json
{
  "production": "models/v20260801_a3b7c901.pkl",
  "staging": "models/v20260808_b4c8d012.pkl",
  "archived": [
    "models/v20260715_e5f9a123.pkl"
  ]
}

Writing models/registry.json


In [16]:
import json
import pandas as pd
from src.serving.pipeline import FraudPipeline

# 1. Sample training batch
X_train = pd.DataFrame({
    'amount': [120.0, 4500.0, 15.5, 890.0],
    'payment_type': ['card', 'wire', 'card', 'crypto'],
    'device_category': ['mobile', 'desktop', 'mobile', 'desktop']
})
y_train = pd.Series([0, 1, 0, 1])

# 2. Fit pipeline
fraud_model = FraudPipeline()
fraud_model.fit(X_train, y_train)

# 3. Save model artifact with automated version string
version_tag, saved_path = fraud_model.save(output_dir="models")

# 4. Update models/registry.json staging path
registry_path = "models/registry.json"
with open(registry_path, "r") as f:
    registry = json.load(f)

registry["staging"] = saved_path

with open(registry_path, "w") as f:
    json.dump(registry, f, indent=2)

print(f"Updated registry staging pointer to: {saved_path}")

Pipeline saved successfully to: models/v20260808_0716ac8.pkl
Updated registry staging pointer to: models/v20260808_0716ac8.pkl


In [18]:
!pip install -q huggingface_hub